# Solution 2.C: Census 2022 Real-Data Pipeline (STATA)

One applied exercise on the South Africa **Census 2022** 10% sample (STATA, about 1.3M household rows). It applies and extends the techniques from **2.2.1 to 2.2.5** on real data.

> Cleans, transforms and merges in memory, then saves one analysis table to `20_processed/`.

### Path Setup and data load

In [ ]:
import os
import numpy as np
import pandas as pd

RAW_DATA_DIR = '../../data/0_raw/south_africa/Census2022SampleSTATA'
hh_path = os.path.join(RAW_DATA_DIR, 'Census2022Households.dta')
geo_path = os.path.join(RAW_DATA_DIR, 'Census2022Geography.dta')
persons_path = os.path.join(RAW_DATA_DIR, 'Census2022Persons.dta')

# Households and geography load in full
hh = pd.read_stata(hh_path)
geo = pd.read_stata(geo_path)

# Persons is large, so load only the first 100k rows with a chunked reader
with pd.read_stata(persons_path, chunksize=100_000) as reader:
    persons = next(reader)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded:', hh.shape, geo.shape, persons.shape)

---

## Task 1: Read the descriptions, pick your columns

`hh` is the full households table. `reader.variable_labels()` returns `{column: description}`: what each of the 32 columns *is*. The names like `DERH_HSIZE` are opaque, the descriptions are not.

**Your job:**

1. Build a list `COLS` with `QID` plus the column matching each description: Household size, Sex of head of the household, Population group of head of the household, Age of head of the household, Toilet facilities, Cellphone, Access to internet, Household weight.
2. Select those columns from `hh` into a new frame `df` (use `.copy()`).
3. The quantities arrived labelled (`DERH_HHAGE` as `'12'..'110'`, `DERH_HSIZE` as `'1'..'9','10 +'`), so convert `DERH_HHAGE` and `DERH_HSIZE` back to numbers (replace `'10 +'` with `'10'` first, then `pd.to_numeric`).

In [ ]:
reader = pd.io.stata.StataReader(hh_path)
var_labels = reader.variable_labels()     # {column: description}

var_labels        # read the descriptions to decide which columns to keep

In [ ]:
COLS = [
    'QID',                   # identifier
    'DERH_HSIZE',            # Household size
    'DERH_HHSEX',            # Sex of head of the household
    'DERH_HHPOP',            # Population group of head of the household
    'DERH_HHAGE',            # Age of head of the household
    'H08_TOILET',            # Toilet facilities
    'H12_CELLPHONE',         # Cellphone
    'H13_INTERNET_ACCESS',   # Access to internet
    'HH_WGT',                # Household weight
]

df = hh[COLS].copy()
# the quantity columns came in as text categories, convert to numbers
df['DERH_HHAGE'] = pd.to_numeric(df['DERH_HHAGE'].astype('object'))
df['DERH_HSIZE'] = pd.to_numeric(df['DERH_HSIZE'].astype('object').replace('10 +', '10'))

print('Shape:', df.shape)
df.head()

**Question:** Why pick columns with `variable_labels()`, and why did `DERH_HHAGE`/`DERH_HSIZE` still need converting?

**Answer:** `variable_labels()` maps each opaque name (`DERH_HSIZE`) to a plain description ("Household size"), so you select by *meaning* rather than guessing. `read_stata` labelled the quantities too: age arrived as text `'12'..'110'` and size with a `'10 +'` top-code, so you convert those back to numbers to bin and compare them. Use labels for true categories, numbers for quantities.

---

## Task 2: Quick diagnostics

Get a feel for the table before transforming it.

**Your job:** show the dtypes and memory use, then a per-column summary that includes the categorical columns (transposed so it reads top to bottom).

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

**Question:** Which columns are `category` and which are numeric, and why is that the split you want?

**Answer:** Category: `DERH_HHSEX`, `DERH_HHPOP`, `H08_TOILET`, `H12_CELLPHONE`, `H13_INTERNET_ACCESS` (true categories, kept as readable labels). Numeric: `DERH_HSIZE`, `DERH_HHAGE`, `HH_WGT` (quantities you compute on, after the Task 1 conversion). That is the split you want: label the categoricals, compute on the numbers.

---

## Task 3: Coded missing values

The labels make missing obvious. `H12_CELLPHONE` and `H13_INTERNET_ACCESS` each have an **`"Unspecified"`** category, which means non-response. Note `H13` also has `"No access to internet services"` and `H08_TOILET` has `"None"`: those are **real** answers, not missing.

**Your job:** replace only `"Unspecified"` with `np.nan` in `H12_CELLPHONE` and `H13_INTERNET_ACCESS`, then count how many values are now missing.

In [ ]:
print(df['H12_CELLPHONE'].value_counts(dropna=False))

df['H12_CELLPHONE'] = df['H12_CELLPHONE'].replace('Unspecified', np.nan)
df['H13_INTERNET_ACCESS'] = df['H13_INTERNET_ACCESS'].replace('Unspecified', np.nan)

print()
print(df[['H12_CELLPHONE', 'H13_INTERNET_ACCESS']].isna().sum())

**Questions:**

- How many became `NaN`?
- Why recode `"Unspecified"` but not `"No access to internet services"` or `"None"`?

**Answers:**

- About 269,433 (cellphone) and 269,375 (internet), the "Unspecified" non-responses.
- `"Unspecified"` means the question was not answered (missing). `"No access to internet services"` and `"None"` (no toilet) are **real answers**, deprivations you want to keep and count. The trap is treating "no access" as missing.

---

## Task 4: Transform with `pd.cut` and `np.select`

Derive two categorical features.

**Your job:**

- `head_age_band`: cut `DERH_HHAGE` with edges `[0, 18, 35, 50, 65, 120]` and labels `'<18', '18-34', '35-49', '50-64', '65+'`.
- `hsize_cat`: with `np.select`, label households `'Small'` (size <= 2), `'Medium'` (3 to 5), `'Large'` (> 5).

In [ ]:
df['head_age_band'] = pd.cut(df['DERH_HHAGE'], bins=[0, 18, 35, 50, 65, 120],
                             labels=['<18', '18-34', '35-49', '50-64', '65+'])

df['hsize_cat'] = np.select(
    [df['DERH_HSIZE'] <= 2, df['DERH_HSIZE'] <= 5, df['DERH_HSIZE'] > 5],
    ['Small', 'Medium', 'Large'], default='Unknown')

df[['QID', 'DERH_HHAGE', 'head_age_band', 'DERH_HSIZE', 'hsize_cat']].head()

**Question:** Which band does age 18 fall into?

**Answer:** `'<18'`. `pd.cut` intervals are right-closed by default, so `(0, 18]` includes 18. Pass `right=False` (or move the boundary to 19) if you want 18 to count as an adult.

---

## Task 5: Subset with boolean indexing

Boolean indexing `df[condition]` keeps only the matching rows. Combine conditions with `&`/`|`, each comparison in its own parentheses.

**Your job:** count households with no cellphone, and (separately) large households (size >= 6) with no cellphone.

In [ ]:
no_phone = df[df['H12_CELLPHONE'] == 'No']
print('Households with no cellphone:', len(no_phone))

large_no_phone = df[(df['H12_CELLPHONE'] == 'No') & (df['DERH_HSIZE'] >= 6)]
print('Large households (6+) with no cellphone:', len(large_no_phone))

**Question:** What goes wrong without the inner parentheses?

**Answer:** `&` binds tighter than `==`/`>=`, so `'No' & df['DERH_HSIZE']` is evaluated first and raises a `TypeError`. Wrap each comparison: `(df['H12_CELLPHONE'] == 'No') & (df['DERH_HSIZE'] >= 6)`.

---

## Task 6: Transform with `.loc` and `assign`

Two more derived columns.

**Your job:**

- `child_headed`: a boolean, `True` where the head is under 18. Start it `False`, then set the matching rows with `.loc`.
- With `df.assign(...)`, add `has_internet` (`np.nan` where `H13_INTERNET_ACCESS` is missing, otherwise `True`/`False` for "not 'No access to internet services'") and `internet_label` ('Has internet' where `has_internet == 1`, else 'No internet').

In [ ]:
df['child_headed'] = False
df.loc[df['DERH_HHAGE'] < 18, 'child_headed'] = True
print('Child-headed households:', df['child_headed'].sum())

In [ ]:
df = df.assign(
    has_internet = lambda x: np.where(
        x['H13_INTERNET_ACCESS'].isna(), np.nan,
        x['H13_INTERNET_ACCESS'] != 'No access to internet services'),
    internet_label = lambda x: np.where(x['has_internet'] == 1, 'Has internet', 'No internet'),
)
df[['QID', 'H13_INTERNET_ACCESS', 'has_internet', 'internet_label']].head()

**Question:** Why can `internet_label` reference `x['has_internet']` created moments earlier?

**Answer:** Inside `assign`, each `lambda x:` receives the DataFrame as it exists at that point in the call, so `internet_label` sees `has_internet`, defined one line earlier in the same `assign`. The steps build up left to right.

---

## Task 7: Merge I, geography (one-to-one)

`geo` is already loaded (`QID, Province, District, Municipality, Geo_type`). Attach it, then add a hand-built zone lookup.

**Your job:**

1. Rename `Province` to `province_name` in `geo`, and cast `geo['QID']` to string.
2. Left-merge `geo` onto `df` on `QID`. Validate one-to-one and pass `indicator=True` to confirm every row matched.
3. Merge the provided `region_lookup` to add a `zone` column (the key has a different name on each side).

In [ ]:
geo = geo.rename(columns={'Province': 'province_name'})   # already labelled 'Western Cape', ...
geo['QID'] = geo['QID'].astype(str)                        # key hygiene: match dtype on both sides

df = pd.merge(df, geo, on='QID', how='left', validate='one_to_one', indicator=True)
print(df['_merge'].value_counts())
df = df.drop(columns='_merge')

In [ ]:
region_lookup = pd.DataFrame({
    'prov': ['Western Cape', 'Eastern Cape', 'Northern Cape', 'KwaZulu-Natal',
             'Free State', 'North West', 'Gauteng', 'Mpumalanga', 'Limpopo'],
    'zone': ['Coastal', 'Coastal', 'Coastal', 'Coastal',
             'Inland', 'Inland', 'Inland', 'Inland', 'Inland'],
})
df = pd.merge(df, region_lookup, left_on='province_name', right_on='prov', how='left')
df = df.drop(columns='prov')
df[['QID', 'province_name', 'zone']].head()

**Question:** Why is a hand-written lookup OK here, but hand-typing labels was not?

**Answer:** `zone` is your own analytical classification: it is not in the file, so you must define it. The province labels already exist inside the `.dta` (you got them for free), so re-typing them by hand would only risk errors. Define what is not there, reuse what is. (`left_on`/`right_on` join because the key has a different name on each side.)

---

## Task 8: Transform strings with a lookup (`apply` with extra args)

`Series.apply(func, args=(...))` runs your function on each value and passes extra arguments too. Build `province_code(name, overrides)` that returns a short code: by default the first three letters of `name` uppercased, but a value from the `overrides` dict for the names where that rule fails (collisions or well-known abbreviations). Apply it to `province_name` to add a `province_code` column.

In [ ]:
PROVINCE_CODES = {
    'KwaZulu-Natal': 'KZN',
    'North West': 'NW',
    'Northern Cape': 'NC',
}

def province_code(name, overrides):
    # a lookup for the tricky names, otherwise the first three letters uppercased
    return overrides.get(name, name[:3].upper())

df['province_code'] = df['province_name'].apply(province_code, args=(PROVINCE_CODES,))
df[['province_name', 'province_code']].drop_duplicates().sort_values('province_name')

**Question:** Why pass `PROVINCE_CODES` through `args=` instead of hard-coding it inside the function, and what does the override dict fix that the first-three-letters rule alone gets wrong?

**Answer:** Passing it through `args=` keeps the function general: the same `province_code` works with any code table you hand it, and the table lives outside the function where it is easy to edit. The first-three-letters rule collides for `North West` and `Northern Cape` (both would give `NOR`), so the override dict pins those (and maps `KwaZulu-Natal` to the familiar `KZN`).

---

## Task 9: `apply` over rows (`axis=1`)

`df.apply(func, axis=1)` hands **each row** (a Series) to your function, which is needed when the result depends on several columns together. **Build** `deprivation_score(row)` that counts how many of three basic services a household lacks: a cellphone (`H12_CELLPHONE == 'No'`), internet (`has_internet == 0`), and a toilet (`H08_TOILET == 'None'`). Row-wise `apply` runs Python once per row, so it is slow on millions of rows: we demo it on a sample.

In [ ]:
def deprivation_score(row):
    # Count how many basic services a household lacks (0 to 3)
    checks = {
        'no_cellphone': row['H12_CELLPHONE'] == 'No',
        'no_internet':  row['has_internet'] == 0,
        'no_toilet':    row['H08_TOILET'] == 'None',
    }
    score = 0
    for lacking in checks.values():            # loop over the checks
        if lacking:
            score += 1
    return score

sample = df.sample(5000, random_state=0)
sample['deprivation_score'] = sample.apply(deprivation_score, axis=1)
sample['deprivation_score'].value_counts().sort_index()

**Question:** Why is row-wise `apply` (`axis=1`) slow on the full 1.3M rows, and what vectorised alternative could replace this function?

**Answer:** `axis=1` calls the Python function once per row with no vectorisation, so cost scales with the row count, which is why we sample. The same score is far faster built from vectorised boolean columns, e.g. `df['H12_CELLPHONE'].eq('No').astype(int) + df['has_internet'].eq(0).astype(int) + df['H08_TOILET'].eq('None').astype(int)`.

---

## Task 10: Merge II, aggregate persons then many-to-one

`persons` is already loaded (the first 100k rows). Aggregate it to one row per `QID`, then merge back.

**Your job:**

1. (given) Cast `persons['QID']` to string and convert `P04_AGE` to numbers.
2. Group by `QID` to build `hh_summary` with `n_persons` (count of people) and `mean_age` (their mean age).
3. Left-merge `hh_summary` onto `df` (validate one-to-one). Because only 100k person rows were loaded, `n_persons` exists for some households only, so `size_mismatch` is computed where a count exists.
4. Separately, attach each person's `province_name` from `df` with a many-to-one merge.

In [ ]:
persons['QID'] = persons['QID'].astype(str)
persons['P04_AGE'] = pd.to_numeric(persons['P04_AGE'].astype('object'))   # age came in labelled too

hh_summary = persons.groupby('QID').agg(
    n_persons=('PID', 'size'),
    mean_age=('P04_AGE', 'mean'),
).reset_index()

df = pd.merge(df, hh_summary, on='QID', how='left', validate='one_to_one')
# only the first 100k persons were loaded, so compare size where a person count exists
counted = df['n_persons'].notna()
df['size_mismatch'] = counted & (df['DERH_HSIZE'] != df['n_persons'])
print('Households matched to persons:', counted.sum())
print('Reported size != counted persons:', df['size_mismatch'].sum())
df[['QID', 'DERH_HSIZE', 'n_persons', 'mean_age']].head()

In [ ]:
persons_geo = pd.merge(persons, df[['QID', 'province_name']], on='QID', how='left',
                       validate='many_to_one')
print('Persons:', len(persons), '-> after merge:', len(persons_geo))

**Question:** Why `many_to_one` here vs `one_to_one` for geography?

**Answer:** Geography has exactly one row per `QID` (one-to-one with households). The persons file repeats each `QID` (many people per household), so attaching the one household's province to each person is many-to-one. The aggregated `hh_summary` is unique per `QID`, so merging it back is one-to-one. (Only the first 100k persons were loaded, so `n_persons` covers some households only; where it exists, the `10` top-code on household size makes the largest matched households mismatch.)

---

## Task 11: Post-merge validation, then append with `concat`

Check the merged table, then practise stacking rows.

**Your job:** print the number of duplicate `QID`s and the share of rows missing `province_name`. Then take the Gauteng and Western Cape subsets and stack them with `pd.concat` (reset the index).

In [ ]:
print('Rows:', len(df))
print('Duplicate QID:', df['QID'].duplicated().sum())
print('Unmatched geography:', round(df['province_name'].isna().mean(), 4))

In [ ]:
gauteng = df[df['province_name'] == 'Gauteng']
wcape = df[df['province_name'] == 'Western Cape']
stacked = pd.concat([gauteng, wcape], ignore_index=True)
print(len(gauteng), '+', len(wcape), '=', len(stacked))

**Question:** What should you check about inputs before stacking real waves?

**Answer:** Every input needs the **same column names and dtypes**. `concat` aligns on names and silently fills mismatches with `NaN`, so a renamed or missing column becomes a half-empty column. Add a provenance column (e.g. `wave='2022'`) before stacking.

---

## Task 12: Save the analysis table

Write the finished table to CSV under `20_processed/`, then read it back to confirm.

**Your job:** save `df` to `out_path` as CSV without the index, then reload it (keeping `QID` as a string).

In [ ]:
df = df.reset_index(drop=True)
PROC = '../../data/20_processed'
os.makedirs(PROC, exist_ok=True)
out_path = os.path.join(PROC, 'census2022_household_analysis_226.csv')

df.to_csv(out_path, index=False)
print('Saved:', out_path, '|', df.shape)

In [ ]:
check = pd.read_csv(out_path, dtype={'QID': str})
print('Reloaded:', check.shape)
check.head()

**Question:** What does saving to CSV tell you about its limits?

**Answer:** The category labels survive only because they were already text, but the *category dtype* itself is gone (it reloads as plain `object`) and numeric dtypes widen. CSV stores values, not types or label metadata. Parquet (or the original `.dta`) preserves dtypes and labels.